# 06 — Combined portfolio backtest

Combine strategy-level return streams using inverse-vol weighting + optional vol
targeting + drawdown de-risking.

**Inputs:** daily PnL series from each backtest (saved into the DB by the engine).

In [ ]:
import sys, os, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from src.data.yfinance_client import YFinanceClient
from src.strategies.spy_vol_target import SPYVolTargetStrategy, VolTargetConfig
from src.strategies.tsmom import TSMOMStrategy, TSMOMConfig
from src.backtest.engine import run_backtest, EngineConfig

yf = YFinanceClient()
universe = ['SPY','QQQ','IWM','TLT','IEF','GLD','SLV','USO','VNQ','XLE','XLK']
data = {s: yf.get_daily_bars(s, start='2010-01-01')['adj_close'] for s in universe}
prices = pd.DataFrame(data).ffill().dropna()
print('Universe panel:', prices.shape)

In [ ]:
# strategy 1: SPY vol-target
strat1 = SPYVolTargetStrategy(VolTargetConfig(symbol='SPY', target_vol=0.10, rv_window=21))
r1 = run_backtest(strat1, prices[['SPY']], EngineConfig(strategy_name='svt', run_preflight=False, benchmark=None))

# strategy 2: TSMOM
strat2 = TSMOMStrategy(TSMOMConfig(universe=list(prices.columns), per_asset_vol=0.08,
                                    max_gross_leverage=1.0, allow_short=True))
r2 = run_backtest(strat2, prices, EngineConfig(strategy_name='tsmom', rebalance_freq='M', run_preflight=False, benchmark=None))

streams = pd.DataFrame({'spy_vt': r1['returns'], 'tsmom': r2['returns']}).fillna(0)
print('Streams:', streams.shape)
print('Per-stream Sharpe:')
for c in streams.columns:
    s = streams[c]
    print(f'  {c}: Sharpe={s.mean()/s.std()*np.sqrt(252):+.2f}')

### Inverse-vol allocation

In [ ]:
ivol = 1.0 / streams.std()
w = ivol / ivol.sum()
print('Weights:'); print(w.to_string())
combo = (streams * w).sum(axis=1)
print(f'\nCombo Sharpe: {combo.mean()/combo.std()*np.sqrt(252):+.2f}')
print(f'Combo MDD: {((1+combo).cumprod() / (1+combo).cumprod().cummax() - 1).min():+.2%}')

### Correlations + rolling correlations

In [ ]:
print('Static corr:'); print(streams.corr().round(2))
rc = streams['spy_vt'].rolling(126).corr(streams['tsmom'])
fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(rc.index, rc.values, linewidth=0.7)
ax.axhline(0, color='gray', linewidth=0.5)
ax.set_title('Rolling 126d correlation: SPY-VT vs TSMOM')
ax.grid(alpha=0.3); plt.show()

### Combined equity

In [ ]:
eq = (1 + combo).cumprod() * 100_000
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(eq.index, eq.values, linewidth=1.2, label='combined')
ax.plot(r1['equity'].index, r1['equity'].values, linewidth=0.7, alpha=0.6, label='spy_vt')
ax.plot(r2['equity'].index, r2['equity'].values, linewidth=0.7, alpha=0.6, label='tsmom')
ax.set_title('Inverse-vol combined portfolio')
ax.grid(alpha=0.3); ax.legend(); plt.show()